# 7.3 隐私与安全

端侧部署大模型涉及用户数据隐私和模型知识产权保护。核心技术包括联邦学习、差分隐私、模型水印和安全推理。在产业级部署中，隐私安全不仅是合规要求，更是用户信任的基础。

## 什么是隐私与安全技术？

隐私与安全技术保护用户数据不被泄露，同时保护模型知识产权不被窃取。核心方法包括：
- **联邦学习**：数据不出本地，仅共享模型更新
- **差分隐私**：在数据/梯度中添加噪声，使个体数据无法被推断
- **模型水印**：在模型中嵌入可验证的标识，证明所有权
- **安全推理**：使用同态加密/安全多方计算，在加密数据上推理
- **模型提取防御**：防止攻击者通过API查询逆向重建模型

## 为什么需要隐私与安全技术？

1. **法规要求**：GDPR、CCPA、《个人信息保护法》等法规要求保护用户数据
2. **模型窃取风险**：API部署的模型可能被逆向工程窃取
3. **端侧数据敏感**：端侧处理的数据（健康、位置、通信）通常更敏感
4. **成员推断攻击**：攻击者可判断某样本是否在训练集中
5. **数据投毒**：恶意训练数据可植入后门

## 隐私与安全的数学原理

**差分隐私**：机制$\mathcal{M}$满足$(\varepsilon, \delta)$-差分隐私，当且仅当对任意相邻数据集$D, D'$和任意输出集合$S$：
$$\Pr[\mathcal{M}(D) \in S] \leq e^{\varepsilon} \cdot \Pr[\mathcal{M}(D') \in S] + \delta$$
其中$\varepsilon$为隐私预算（越小越隐私），$\delta$为失败概率

**联邦学习（FedAvg）**：$K$个客户端的聚合更新：
$$\theta_{t+1} = \sum_{i=1}^{K} \frac{n_i}{N} \theta_{t+1}^{(i)}$$
其中$n_i$为第$i$个客户端的数据量，$N = \sum n_i$

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")

### 联邦学习（Federated Learning）

#### 什么是联邦学习？

在端侧本地训练模型更新，仅上传梯度/参数差值到服务器聚合，原始数据不出端。FedAvg是最经典的联邦学习算法。

#### 为什么用联邦学习？

1. **数据隐私**：原始数据不出端侧，仅上传模型更新
2. **法规合规**：满足GDPR等数据保护法规
3. **数据多样性**：多端数据联合训练，模型泛化性更好

#### 联邦学习的数学原理（FedAvg）

$K$个客户端的聚合更新：
$$\theta_{t+1} = \sum_{i=1}^{K} \frac{n_i}{N} \theta_{t+1}^{(i)}$$

其中：
- $\theta_{t+1}^{(i)}$：第$i$个客户端本地训练后的模型参数
- $n_i$：第$i$个客户端的数据量
- $N = \sum_{i=1}^{K} n_i$：总数据量
- 聚合权重$\frac{n_i}{N}$：数据量越大的客户端权重越高

#### 联邦学习的核心挑战

1. **Non-IID数据**：不同客户端的数据分布差异大，导致模型发散
   - 解决：FedProx（近端正则化）、SCAFFOLD（方差缩减）、FedNova（归一化平均）
2. **通信效率**：上传完整模型更新代价高
   - 解决：梯度压缩（Top-K稀疏化）、量化（INT8上传）、LoRA联邦微调
3. **安全聚合**：服务器不应看到单个客户端的更新
   - 解决：Secret Sharing + Secure Aggregation（Google的SecAgg协议）
4. **掉线客户端**：部分客户端可能掉线，需要容错聚合
   - 解决：设置最小参与数$K_{\min}$，超时后用已收到的更新聚合

In [ ]:
class FederatedClient:
    """联邦学习客户端"""
    def __init__(self, client_id, model, data, lr=0.01, local_epochs=5):
        self.client_id = client_id
        self.model = model
        self.data = data
        self.lr = lr
        self.local_epochs = local_epochs

    def train_local(self):
        """本地训练"""
        original_params = {n: p.clone() for n, p in self.model.named_parameters()}
        optimizer = torch.optim.SGD(self.model.parameters(), lr=self.lr)
        for _ in range(self.local_epochs):
            for x, y in self.data:
                loss = F.cross_entropy(self.model(x), y)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        updates = {}
        for n, p in self.model.named_parameters():
            updates[n] = original_params[n] - p.data
        return updates

class FederatedServer:
    """联邦学习服务器"""
    def __init__(self, global_model):
        self.global_model = global_model

    def aggregate(self, client_updates, n_samples):
        """FedAvg聚合"""
        total_samples = sum(n_samples)
        with torch.no_grad():
            for name, param in self.global_model.named_parameters():
                weighted_update = torch.zeros_like(param)
                for updates, n in zip(client_updates, n_samples):
                    weighted_update += updates[name] * (n / total_samples)
                param.data -= weighted_update

class SimpleModel(nn.Module):
    def __init__(self, dim=32, n_classes=5):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, 64), nn.ReLU(), nn.Linear(64, n_classes))

    def forward(self, x):
        return self.net(x)

dim, n_classes = 32, 5
global_model = SimpleModel(dim, n_classes)
server = FederatedServer(global_model)

n_clients = 3
clients = []
for i in range(n_clients):
    local_model = SimpleModel(dim, n_classes)
    local_model.load_state_dict(global_model.state_dict())
    x = torch.randn(32 + i * 8, dim)
    y = torch.randint(0, n_classes, (32 + i * 8,))
    data = [(x, y)]
    clients.append(FederatedClient(i, local_model, data, lr=0.01, local_epochs=3))

n_rounds = 5
for round_idx in range(n_rounds):
    client_updates = []
    n_samples = []
    for client in clients:
        client.model.load_state_dict(global_model.state_dict())
        updates = client.train_local()
        client_updates.append(updates)
        n_samples.append(client.data[0][0].shape[0])
    server.aggregate(client_updates, n_samples)

x_test = torch.randn(64, dim)
y_test = torch.randint(0, n_classes, (64,))
with torch.no_grad():
    acc = (global_model(x_test).argmax(1) == y_test).float().mean()

print(f"=== 联邦学习 ===")
print(f"客户端数: {n_clients}")
print(f"通信轮数: {n_rounds}")
print(f"全局模型准确率: {acc:.4f}")

print(f"\n--- 联邦学习变体 ---")
variants = [
    ('FedAvg', '按样本量加权平均', 'IID数据', '基础算法'),
    ('FedProx', '近端正则化$\mu\|\theta-\theta_g\|^2$', 'Non-IID数据', '防止客户端漂移'),
    ('SCAFFOLD', '方差缩减（控制变量）', 'Non-IID数据', '收敛更快'),
    ('FedNova', '归一化本地步数', '异构计算', '公平聚合'),
    ('FedLoRA', '仅上传LoRA更新', '通信受限', '通信量降低90%+'),
]
print(f"\n{'算法':<12} {'核心思想':<30} {'适用场景':<15} {'优势':<20}")
print("-" * 77)
for name, idea, scene, advantage in variants:
    print(f"{name:<12} {idea:<30} {scene:<15} {advantage:<20}")

print(f"\n联邦学习核心:")
print(f"1. 原始数据不出端: 仅上传模型更新")
print(f"2. FedAvg聚合: 按样本量加权平均")
print(f"3. Non-IID挑战: 客户端数据分布差异大，需特殊处理")
print(f"4. 安全聚合: SecAgg协议确保服务器无法看到单个客户端更新")

### 差分隐私（Differential Privacy）

#### 什么是差分隐私？

在数据或梯度中添加精心校准的噪声，使得无法从输出推断任何个体数据的存在与否。提供数学上可证明的隐私保护。

#### 为什么用差分隐私？

1. **数学保证**：提供可证明的隐私保护，不依赖启发式方法
2. **抗成员推断攻击**：攻击者无法判断某个样本是否在训练集中
3. **组合性**：多次查询的隐私损失可组合计算（RDP/Moments Accountant）

#### 差分隐私的数学原理

机制$\mathcal{M}$满足$(\varepsilon, \delta)$-差分隐私，当且仅当对任意相邻数据集$D, D'$和任意输出集合$S$：
$$\Pr[\mathcal{M}(D) \in S] \leq e^{\varepsilon} \cdot \Pr[\mathcal{M}(D') \in S] + \delta$$

其中：
- $\varepsilon$：隐私预算，越小越隐私（通常取0.1-10）
- $\delta$：失败概率，通常取$\frac{1}{|D|^2}$
- 相邻数据集：仅相差一条记录的数据集

#### DP-SGD（差分隐私随机梯度下降）

核心三步：
1. **Per-Sample梯度裁剪**：$\|g_i\|_2 \leq C$，限制单个样本对梯度的贡献
2. **聚合与噪声添加**：$\tilde{g} = \frac{1}{B}(\sum_i \text{clip}(g_i, C)) + \mathcal{N}(0, \sigma^2 C^2 I / B^2)$
3. **隐私会计**：使用RDP（Rényi Differential Privacy）精确跟踪隐私损失

注意：标准实现需要per-sample梯度（而非per-parameter裁剪），这是DP-SGD的关键区别。

#### 隐私预算选择指南

| $\varepsilon$ | 隐私级别 | 精度损失 | 适用场景 |
|-----------|---------|---------|----------|
| < 1 | 极强隐私 | 显著 | 高敏感数据（医疗） |
| 1-3 | 强隐私 | 中等 | 金融、个人数据 |
| 3-8 | 中等隐私 | 轻微 | 一般用户数据 |
| 8-20 | 弱隐私 | 极小 | 低敏感数据 |
| > 20 | 弱保护 | 几乎无 | 非敏感场景 |

In [ ]:
class DPSGD:
    """差分隐私SGD优化器
    注意：本实现使用per-parameter梯度裁剪作为简化演示。
    严格的DP-SGD需要per-sample梯度裁剪（使用Opacus等库），
    即对每个样本单独计算梯度并裁剪，而非对整个batch的梯度裁剪。
    per-parameter裁剪不满足严格的差分隐私保证。"""
    def __init__(self, model, lr=0.01, noise_multiplier=1.0, max_grad_norm=1.0):
        self.model = model
        self.lr = lr
        self.noise_multiplier = noise_multiplier
        self.max_grad_norm = max_grad_norm

    def step(self, loss):
        """差分隐私梯度更新"""
        loss.backward()
        total_norm = torch.nn.utils.clip_grad_norm_(
            self.model.parameters(), self.max_grad_norm)
        for p in self.model.parameters():
            if p.grad is not None:
                noise = torch.randn_like(p.grad) * self.noise_multiplier * self.max_grad_norm
                p.data -= self.lr * (p.grad + noise)
                p.grad = None

    def compute_epsilon(self, n_steps, delta=1e-5, batch_size=32, dataset_size=1000):
        """估算隐私预算ε（简化公式，严格计算需RDP/Moments Accountant）"""
        q = batch_size / dataset_size
        sigma = self.noise_multiplier
        epsilon = q * np.sqrt(2 * n_steps * np.log(1/delta)) / sigma
        return epsilon

model_dp = SimpleModel(dim=32, n_classes=5)
dp_optimizer = DPSGD(model_dp, lr=0.01, noise_multiplier=1.0, max_grad_norm=1.0)

x = torch.randn(64, 32)
y = torch.randint(0, 5, (64,))

for _ in range(20):
    loss = F.cross_entropy(model_dp(x), y)
    dp_optimizer.step(loss)

with torch.no_grad():
    acc_dp = (model_dp(x).argmax(1) == y).float().mean()

model_no_dp = SimpleModel(dim=32, n_classes=5)
optimizer = torch.optim.SGD(model_no_dp.parameters(), lr=0.01)
for _ in range(20):
    loss = F.cross_entropy(model_no_dp(x), y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

with torch.no_grad():
    acc_no_dp = (model_no_dp(x).argmax(1) == y).float().mean()

epsilon = dp_optimizer.compute_epsilon(n_steps=20, batch_size=32, dataset_size=1000)

print(f"=== 差分隐私训练 ===")
print(f"噪声倍数(σ): {dp_optimizer.noise_multiplier}")
print(f"梯度裁剪(C): {dp_optimizer.max_grad_norm}")
print(f"隐私预算ε: {epsilon:.4f}")
print(f"\n准确率对比:")
print(f"  无DP: {acc_no_dp:.4f}")
print(f"  有DP: {acc_dp:.4f}")
print(f"  精度损失: {(acc_no_dp - acc_dp)*100:.2f}%")

print(f"\n--- 不同噪声倍数的影响 ---")
for sigma in [0.5, 1.0, 2.0, 5.0]:
    eps = DPSGD(SimpleModel(), noise_multiplier=sigma).compute_epsilon(
        n_steps=20, batch_size=32, dataset_size=1000)
    print(f"  σ={sigma}: ε={eps:.2f} ({'强隐私' if eps < 3 else '中等隐私' if eps < 8 else '弱隐私'})")

print(f"\n产业级DP-SGD实践:")
print(f"1. 使用Opacus库: 自动per-sample梯度裁剪，严格DP保证")
print(f"2. 隐私会计: 使用RDP（Rényi DP）精确跟踪ε，而非简化公式")
print(f"3. 典型设置: ε=3~8, δ=1e-5, σ=0.5~2.0")
print(f"4. DP-FedAvg: 联邦学习+差分隐私，在聚合后添加噪声")

### 模型水印与知识产权保护

#### 什么是模型水印？

在模型中嵌入可验证的标识信息，用于证明模型的所有权和来源。当模型被窃取或非法复制时，可以通过水印验证所有权。

#### 为什么需要模型水印？

1. **模型窃取风险**：API部署的模型可能被逆向工程窃取（通过大量查询训练替代模型）
2. **知识产权保护**：训练模型投入大量资源，需要保护投资
3. **溯源追踪**：泄露的模型可以追溯到源头

#### 水印方法分类

| 方法 | 原理 | 鲁棒性 | 隐蔽性 | 适用场景 |
|------|------|--------|--------|----------|
| 后门水印 | 植入触发模式→特定输出 | 中等 | 低 | 版权声明 |
| 权重水印 | 在权重中嵌入信息 | 高 | 高 | 溯源追踪 |
| 指纹水印 | 模型输出分布特征 | 低 | 高 | 相似度检测 |
| 数据集水印 | 训练数据中植入标记 | 中等 | 中等 | 数据溯源 |

#### 模型水印的数学原理

**后门水印**：在训练时植入特定触发模式：
$$f(x_{\text{trigger}}) = y_{\text{watermark}}$$

其中$x_{\text{trigger}}$为特定的触发输入，$y_{\text{watermark}}$为预设的水印输出。

**权重水印**：在权重矩阵中嵌入信息：
$$W' = W + \epsilon \cdot \text{sign}(W) \odot M$$

其中$M$为水印掩码，$\epsilon$为嵌入强度，$\text{sign}(W)$保留权重符号。

验证：计算$\text{corr}(W', M)$，超过阈值则水印存在。

#### 模型提取攻击与防御

**攻击**：攻击者通过大量API查询收集输入-输出对，训练替代模型
- 查询次数：$O(N \cdot d)$，$N$为样本数，$d$为输出维度
- 防御难度：攻击者只需输入-输出对，无需访问模型内部

**防御**：
- **输出扰动**：添加少量噪声到输出，降低替代模型精度
- **查询限制**：限制单个用户的API调用频率
- **水印验证**：在替代模型上验证水印，证明窃取行为
- **输出截断**：返回top-K而非完整概率分布

In [ ]:
class ModelWatermark:
    """模型水印: 在权重中嵌入不可见标识"""
    def __init__(self, key=42):
        self.key = key

    def embed(self, model, strength=0.001):
        """嵌入水印"""
        torch.manual_seed(self.key)
        watermark_bits = torch.randint(0, 2, (1024,))
        bit_idx = 0
        for name, param in model.named_parameters():
            if param.dim() >= 2:
                n_embed = min(param.numel() // 4, watermark_bits.shape[0] - bit_idx)
                if n_embed <= 0:
                    continue
                flat = param.data.flatten()
                for i in range(n_embed):
                    idx = i + bit_idx
                    if idx >= watermark_bits.shape[0]:
                        break
                    bit = watermark_bits[idx]
                    flat[i * 4] += strength * (1 if bit else -1)
                param.data = flat.reshape(param.shape)
                bit_idx += n_embed
        return watermark_bits

    def verify(self, model, watermark_bits, strength=0.001):
        """验证水印"""
        torch.manual_seed(self.key)
        bit_idx = 0
        correct = 0
        total = 0
        for name, param in model.named_parameters():
            if param.dim() >= 2:
                n_embed = min(param.numel() // 4, watermark_bits.shape[0] - bit_idx)
                if n_embed <= 0:
                    continue
                flat = param.data.flatten()
                for i in range(n_embed):
                    idx = i + bit_idx
                    if idx >= watermark_bits.shape[0]:
                        break
                    detected = flat[i * 4] > 0
                    expected = watermark_bits[idx].bool()
                    correct += (detected == expected).float().item()
                    total += 1
                bit_idx += n_embed
        return correct / max(total, 1)

model = SimpleModel(dim=32, n_classes=5)
watermarker = ModelWatermark(key=42)
bits = watermarker.embed(model, strength=0.01)
verify_rate = watermarker.verify(model, bits, strength=0.01)

x = torch.randn(32, 32)
y = torch.randint(0, 5, (32,))
with torch.no_grad():
    acc = (model(x).argmax(1) == y).float().mean()

print(f"=== 模型水印 ===")
print(f"水印验证率: {verify_rate:.2%}")
print(f"模型准确率: {acc:.4f} (水印对精度影响极小)")

print(f"\n--- 水印鲁棒性测试 ---")
model_finetuned = SimpleModel(dim=32, n_classes=5)
model_finetuned.load_state_dict(model.state_dict())
optimizer = torch.optim.SGD(model_finetuned.parameters(), lr=0.001)
for _ in range(10):
    loss = F.cross_entropy(model_finetuned(x), y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
verify_after_ft = watermarker.verify(model_finetuned, bits, strength=0.01)
print(f"微调后水印验证率: {verify_after_ft:.2%}")
print(f"(微调可能部分破坏水印，但权重水印通常比后门水印更鲁棒)")

print(f"\n=== 隐私安全技术总结 ===")
print(f"\n{'技术':<15} {'保护对象':<15} {'核心机制':<25} {'产业级工具':<20}")
print("-" * 75)
techniques = [
    ('联邦学习', '数据隐私', '数据不出端+聚合', 'PySyft/FedML/TFF'),
    ('差分隐私', '数据隐私', '噪声添加+裁剪', 'Opacus/DP-SGD'),
    ('模型水印', '模型IP', '权重/后门嵌入', '自定义实现'),
    ('安全推理', '数据+模型', '同态加密/MPC', 'TenSEAL/MP-SPDZ'),
    ('提取防御', '模型IP', '输出扰动+限流', 'API Gateway'),
]
for name, target, mechanism, tool in techniques:
    print(f"{name:<15} {target:<15} {mechanism:<25} {tool:<20}")

---
### 可信执行环境（TEE）与AI

可信执行环境（Trusted Execution Environment, TEE）是处理器中的安全区域，保证内部代码和数据的机密性与完整性，即使操作系统被攻破也无法窃取TEE内的数据。

#### 主流TEE方案

| TEE方案 | 厂商 | 安全内存 | AI适用性 | 代表设备 |
|---------|------|---------|---------|----------|
| **ARM TrustZone** | ARM | 数百KB | 低（内存太小） | 大多数Android |
| **Intel SGX** | Intel | ~128MB EPC | 中（小模型） | Intel CPU |
| **Intel TDX** | Intel | VM级 | 高（大模型） | Intel Xeon |
| **AMD SEV-SNP** | AMD | VM级 | 高（大模型） | AMD EPYC |
| **Apple Secure Enclave** | Apple | 数MB | 低（仅密钥） | Apple Silicon |
| **Android TEE** | 多厂商 | 数MB | 低（仅密钥） | Android手机 |

#### TEE在AI中的应用

1. **模型IP保护**：模型权重在TEE内解密和推理，外部无法读取
2. **用户数据保护**：用户输入在TEE内处理，服务器管理员也无法查看
3. **联邦学习安全聚合**：在TEE内聚合客户端更新，防止恶意服务器
4. **模型水印验证**：在TEE内验证水印，防止验证过程被篡改

#### TEE的性能开销

| 操作 | 非TEE | SGX TEE | 开销 |
|------|-------|---------|------|
| 内存访问 | ~100ns | ~500ns | 5x |
| 系统调用 | ~1μs | ~10μs | 10x |
| LLM推理(7B) | 50ms/tok | 80-120ms/tok | 1.6-2.4x |

开销来源：
- **EPC换页**：SGX的Enclave Page Cache(EPC)仅128MB，大模型需要频繁换页
- **加密开销**：TEE内存进出需要加密/解密
- **上下文切换**：TEE↔非TEE的切换开销

#### 产业实践：Confidential Computing

- **Azure Confidential Computing**：基于Intel SGX/TDX的AI推理服务
- **Google Confidential VMs**：基于AMD SEV-SNP的云端推理
- **端侧TEE**：目前端侧TEE内存太小，无法运行LLM，仅用于密钥管理和模型解密

In [ ]:
class TEESimulator:
    """可信执行环境模拟器"""
    def __init__(self):
        self.tee_solutions = {
            'Intel SGX': {'secure_memory_mb': 128, 'mem_overhead': 5, 'ctx_switch_us': 10},
            'Intel TDX': {'secure_memory_mb': 32768, 'mem_overhead': 1.2, 'ctx_switch_us': 2},
            'AMD SEV-SNP': {'secure_memory_mb': 65536, 'mem_overhead': 1.1, 'ctx_switch_us': 1},
            'ARM TrustZone': {'secure_memory_mb': 4, 'mem_overhead': 10, 'ctx_switch_us': 5},
        }

    def estimate_llm_inference(self, model_size_mb, tee_name, base_latency_ms_per_tok=50):
        """估算TEE内LLM推理性能"""
        tee = self.tee_solutions[tee_name]
        fits_memory = model_size_mb <= tee['secure_memory_mb']
        overhead = tee['mem_overhead']
        if not fits_memory:
            n_pages = (model_size_mb - tee['secure_memory_mb']) // 4 + 1
            overhead += n_pages * 0.01
        tee_latency = base_latency_ms_per_tok * overhead
        return {
            'tee': tee_name,
            'fits_memory': fits_memory,
            'overhead_factor': overhead,
            'tee_latency_ms_per_tok': tee_latency,
            'slowdown': tee_latency / base_latency_ms_per_tok,
        }

tee_sim = TEESimulator()
print("=== TEE内LLM推理性能估算 ===")
print(f"\n--- 7B INT4模型 (~3.5GB) ---")
print(f"{'TEE方案':<18} {'安全内存':<12} {'适合内存':<8} {'开销因子':<10} {'延迟(ms/tok)':<14} {'减速比'}")
print("-" * 72)
for tee_name in ['Intel SGX', 'Intel TDX', 'AMD SEV-SNP', 'ARM TrustZone']:
    r = tee_sim.estimate_llm_inference(3500, tee_name)
    fits = '✓' if r['fits_memory'] else '✗'
    print(f"{r['tee']:<18} {tee_sim.tee_solutions[tee_name]['secure_memory_mb']}MB{'':<7} "
          f"{fits:<8} {r['overhead_factor']:<10.1f} {r['tee_latency_ms_per_tok']:<14.1f} {r['slowdown']:.1f}x")

print(f"\n--- 1.5B INT4模型 (~750MB) ---")
for tee_name in ['Intel SGX', 'Intel TDX', 'AMD SEV-SNP']:
    r = tee_sim.estimate_llm_inference(750, tee_name, base_latency_ms_per_tok=15)
    fits = '✓' if r['fits_memory'] else '✗'
    print(f"  {r['tee']}: {fits}, 延迟={r['tee_latency_ms_per_tok']:.1f}ms/tok, 减速{r['slowdown']:.1f}x")

print(f"\n=== 产业实践要点 ===")
print(f"1. 端侧TEE(TrustZone/Secure Enclave)内存太小，无法直接运行LLM")
print(f"2. 端侧TEE的用途: 密钥管理、模型解密、完整性校验")
print(f"3. 云端TEE(TDX/SEV-SNP)可运行完整LLM，开销约1.1-1.5x")
print(f"4. SGX的128MB EPC是大模型推理的瓶颈，需频繁换页")
print(f"5. 未来: ARM CCA(Confidential Computing Architecture)将支持更大安全内存")

---
### 安全推理：同态加密与安全多方计算

安全推理（Secure Inference）允许客户端在加密数据上获得推理结果，服务器无法获知输入数据或模型权重。

#### 同态加密（Homomorphic Encryption, HE）

同态加密允许在密文上直接计算，计算结果解密后等于明文计算的结果：

$$\text{Dec}(	ext{Enc}(a) \oplus \text{Enc}(b)) = a + b$$
$$\text{Dec}(	ext{Enc}(a) \otimes \text{Enc}(b)) = a \times b$$

其中$\oplus$和$\otimes$为密文上的加法和乘法运算。

| HE方案 | 支持运算 | 密文膨胀比 | 推理速度 | 适用场景 |
|--------|---------|-----------|---------|----------|
| **BFV/BGV** | 整数加法+乘法 | 100-1000x | 极慢 | 简单模型 |
| **CKKS** | 近似浮点运算 | 100-1000x | 慢 | 神经网络 |
| **TFHE** | 任意布尔电路 | 10000x+ | 极慢 | 通用计算 |

#### 安全多方计算（Secure Multi-Party Computation, MPC）

MPC允许多方在不泄露各自输入的情况下共同计算函数：

$$\text{Alice}(x) + \text{Bob}(W) \xrightarrow{\text{MPC}} f(x, W) = Wx$$

Alice拥有输入$x$，Bob拥有模型$W$，双方都学不到对方的信息。

| MPC协议 | 通信轮数 | 通信量 | 安全模型 | 适用场景 |
|---------|---------|--------|---------|----------|
| **Garbled Circuit** | $O(1)$ | $O(|C|)$ | 半诚实 | 低延迟场景 |
| **Secret Sharing** | $O(d)$ | $O(d \cdot n)$ | 半诚实/恶意 | 深度网络 |
| **OT-based** | $O(1)$ | $O(n)$ | 半诚实 | 非线性运算 |

#### 安全推理的性能挑战

| 方法 | 1层Transformer延迟 | 32层总延迟 | vs 明文 | 通信量 |
|------|-------------------|-----------|---------|--------|
| **明文** | 1ms | 32ms | 1x | 0 |
| **CKKS HE** | 500ms | 16s | 500x | 100MB/req |
| **2PC Secret Sharing** | 50ms | 1.6s | 50x | 500MB/req |
| **HE+MPC混合** | 100ms | 3.2s | 100x | 200MB/req |

#### 产业实践

- **CipherGPT**：基于GC的GPT推理，通信量优化
- **BumbleBee**：基于HE的Transformer推理，使用SIMD加速
- **Cheetah**：HE+MPC混合方案，非线性用MPC，线性用HE
- **当前局限**：安全推理的延迟和通信量仍远超明文，仅适用于高安全场景

In [ ]:
class SecureInferenceSimulator:
    """安全推理性能模拟器"""
    def __init__(self):
        self.methods = {
            '明文': {'latency_per_layer_ms': 1, 'comm_mb_per_req': 0, 'security': '无'},
            'CKKS HE': {'latency_per_layer_ms': 500, 'comm_mb_per_req': 3, 'security': '信息论安全'},
            '2PC Secret Sharing': {'latency_per_layer_ms': 50, 'comm_mb_per_req': 15, 'security': '半诚实安全'},
            'HE+MPC混合': {'latency_per_layer_ms': 100, 'comm_mb_per_req': 6, 'security': '半诚实安全'},
            'TEE(TDX)': {'latency_per_layer_ms': 1.5, 'comm_mb_per_req': 0, 'security': '硬件安全'},
        }

    def estimate_inference(self, n_layers=32, n_tokens=100):
        """估算不同安全推理方案的性能"""
        results = {}
        for name, info in self.methods.items():
            prefill_ms = info['latency_per_layer_ms'] * n_layers
            decode_ms = info['latency_per_layer_ms'] * n_layers * n_tokens
            total_ms = prefill_ms + decode_ms
            results[name] = {
                'prefill_ms': prefill_ms,
                'decode_ms': decode_ms,
                'total_ms': total_ms,
                'comm_mb': info['comm_mb_per_req'] * n_layers * (1 + n_tokens),
                'security': info['security'],
            }
        return results

sec_sim = SecureInferenceSimulator()
results = sec_sim.estimate_inference()

print("=== 安全推理方案对比 (7B模型, 32层, 100 token输出) ===")
print(f"\n{'方案':<22} {'Prefill(ms)':<14} {'Decode(ms)':<14} {'总延迟':<14} {'通信量(MB)':<12} {'安全等级'}")
print("-" * 90)
for name, r in results.items():
    if r['total_ms'] > 60000:
        total_str = f"{r['total_ms']/1000:.0f}s"
    else:
        total_str = f"{r['total_ms']:.0f}ms"
    print(f"{name:<22} {r['prefill_ms']:<14.0f} {r['decode_ms']:<14.0f} {total_str:<14} "
          f"{r['comm_mb']:<12.0f} {r['security']}")

print(f"\n=== 产业实践要点 ===")
print(f"1. HE/MPC的延迟是明文的50-500x，仅适用于高安全场景(医疗/金融)")
print(f"2. 通信量是主要瓶颈: 2PC需要500MB+/请求，不适合移动网络")
print(f"3. TEE是当前最实用的方案: 开销仅1.5x，但依赖硬件信任")
print(f"4. HE+MPC混合方案是研究热点: 线性层用HE，非线性用MPC")
print(f"5. 非线性运算(ReLU/LayerNorm)是安全推理的难点: 需要特殊协议")
print(f"6. 未来: 硬件加速(FPGA/ASIC)可能将HE推理加速10-100x")

---
### 平台安全特性与数据隔离

端侧部署需要利用操作系统的安全特性实现数据隔离和沙箱化。

#### iOS安全架构

| 安全特性 | 用途 | AI相关应用 |
|---------|------|-----------|
| **Secure Enclave** | 密钥存储、生物识别 | 模型解密密钥存储 |
| **Data Protection** | 文件加密 | 模型文件加密 |
| **App Sandbox** | 应用隔离 | 模型数据隔离 |
| **Core ML安全** | 模型绑定设备 | 防止模型跨设备复制 |
| **Neural Engine** | ANE推理 | 模型在ANE内执行，CPU无法读取中间结果 |

#### Android安全架构

| 安全特性 | 用途 | AI相关应用 |
|---------|------|-----------|
| **TrustZone/TEE** | 安全世界执行 | 密钥管理、模型解密 |
| **Android Keystore** | 硬件密钥 | 模型加密密钥 |
| **SafetyNet/Play Integrity** | 设备完整性 | 检测Root/模拟器 |
| **App Sandbox** | 应用隔离 | 模型数据隔离 |
| **NNAPI** | 硬件加速 | NPU推理，中间结果不暴露给CPU |

#### 数据隔离策略

1. **模型数据隔离**：模型权重文件存储在应用沙箱内，其他应用无法访问
2. **推理数据隔离**：用户输入和推理结果仅存储在应用私有目录
3. **内存隔离**：推理过程中的中间结果（KV Cache、激活值）不写入共享内存
4. **日志脱敏**：推理日志不记录用户输入和模型输出的原始内容

#### 防逆向工程

| 防御手段 | iOS | Android |
|---------|-----|---------|
| **代码混淆** | Swift混淆+strip | ProGuard/R8 |
| **反调试** | ptrace检测 | debugger检测 |
| **完整性校验** | App Attest | Play Integrity |
| **越狱/Root检测** | 检测Cydia等 | SafetyNet检测 |
| **模型文件保护** | Data Protection+Core ML绑定 | Keystore加密+NDK编译 |

In [ ]:
class PlatformSecurityAnalyzer:
    """平台安全特性分析器"""
    def __init__(self):
        self.platforms = {
            'iOS': {
                'key_storage': 'Secure Enclave',
                'file_encryption': 'Data Protection(AES-256)',
                'model_binding': 'Core ML设备绑定',
                'anti_tamper': 'App Attest',
                'inference_isolation': 'ANE(中间结果不可读)',
                'security_level': '高',
            },
            'Android': {
                'key_storage': 'Keystore(TEE)',
                'file_encryption': 'AES-256(软件)',
                'model_binding': 'Play Integrity',
                'anti_tamper': 'SafetyNet+R8',
                'inference_isolation': 'NNAPI(NPU隔离)',
                'security_level': '中高',
            },
            'Linux(嵌入式)': {
                'key_storage': '文件系统/TPM',
                'file_encryption': 'LUKS/dm-crypt',
                'model_binding': '无标准方案',
                'anti_tamper': '无标准方案',
                'inference_isolation': '进程隔离',
                'security_level': '低中',
            },
        }

    def compare_platforms(self):
        """对比平台安全特性"""
        return self.platforms

analyzer = PlatformSecurityAnalyzer()
platforms = analyzer.compare_platforms()

print("=== 端侧平台安全特性对比 ===")
aspects = ['key_storage', 'file_encryption', 'model_binding', 'anti_tamper', 'inference_isolation', 'security_level']
aspect_names = ['密钥存储', '文件加密', '模型绑定', '防篡改', '推理隔离', '安全等级']

for aspect, name in zip(aspects, aspect_names):
    print(f"\n{name}:")
    for platform, features in platforms.items():
        print(f"  {platform:<15} {features[aspect]}")

print(f"\n=== 产业实践要点 ===")
print(f"1. iOS安全生态最完善: Secure Enclave+Core ML绑定+App Attest")
print(f"2. Android安全依赖设备厂商: 高端机TEE完善，低端机可能缺失")
print(f"3. 嵌入式Linux安全最弱: 需要自建安全方案(TPM+加密文件系统)")
print(f"4. 模型绑定设备是关键: 防止模型被提取后在其他设备上使用")
print(f"5. NPU推理天然隔离: 中间结果在NPU内部，CPU无法直接读取")
print(f"6. 安全是分层防御: 没有单一方案能完全防止攻击，需多层叠加")

---
## 合规框架与隐私审计

产业级AI部署必须满足各国隐私法规要求，合规不是可选项而是上线前提。

### 主要隐私法规对比

| 法规 | 地区 | 核心要求 | 对端侧AI的影响 |
|------|------|---------|---------------|
| **GDPR** | 欧盟 | 数据最小化、用户同意、被遗忘权 | 端侧处理优先、数据不可离设备 |
| **CCPA** | 加州 | 消费者知情权、删除权、拒绝出售权 | 透明数据使用、提供删除机制 |
| **个人信息保护法** | 中国 | 最小必要、单独同意、安全评估 | 敏感数据需单独同意、跨境传输限制 |
| **AI Act** | 欧盟 | 高风险AI系统透明性、可审计性 | 模型决策可解释、日志可审计 |

### 端侧AI的合规优势

端侧AI在合规方面有天然优势：
- **数据不出设备**：满足数据最小化和本地化要求
- **用户控制**：用户可随时删除本地数据
- **减少传输风险**：避免网络传输中的数据泄露
- **审计友好**：端侧处理日志可在本地完整保留

### 隐私合规Checklist

- [ ] 数据分类：确定哪些数据属于个人敏感信息
- [ ] 同意机制：实现明确的用户授权和数据使用说明
- [ ] 数据最小化：仅采集完成任务所需的最少数据
- [ ] 删除机制：支持用户请求删除所有个人数据
- [ ] 端侧优先：尽可能在端侧处理，减少数据上传
- [ ] 加密存储：本地数据加密存储，密钥绑定设备
- [ ] 审计日志：记录所有数据访问和处理操作
- [ ] 隐私影响评估(PIA)：上线前完成隐私影响评估报告

---
## 总结与最佳实践

### 隐私与安全的核心原则

1. **纵深防御**：没有单一方案能完全防攻击，需加密+TEE+模型绑定+审计多层叠加
2. **端侧优先**：数据不出设备是最强的隐私保护，端侧AI有天然合规优势
3. **安全即基础设施**：安全不是功能，是基础设施，必须在设计阶段就考虑
4. **合规是底线**：GDPR/个人信息保护法是上线前提，不是可选的

### 安全技术选择矩阵

| 安全需求 | 推荐方案 | 性能开销 | 实现复杂度 |
|---------|---------|---------|-----------|
| **模型保护** | 模型绑定设备+加密 | 低(<5%) | 中 |
| **数据隐私** | 端侧处理+联邦学习 | 中(通信开销) | 高 |
| **训练隐私** | 差分隐私(ε=3-8) | 中(精度损失) | 中 |
| **推理安全** | TEE+安全通信 | 低-中 | 高 |
| **知识产权** | 模型水印+法律保护 | 极低 | 低 |
| **合规审计** | 日志+PIA+定期审查 | 低 | 中 |